In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import os
from model.autoencoder import Model
from train import Trainer
from data import get_cross_data, load_data
from util import init_seed


# Seed
init_seed(42)

In [ ]:
cache_samples=False
save_samples=False
batch_size = 8
T = 64          # Number of frames (64)
M = 1           # Number of persons
V = 25          # Number of joints
setting = 'cs'  # 'cs' or 'cv'
dataset = 'ntu120' # 'ntu' or 'ntu120'
lr = 1e-4
train_samples = 64
test_samples = 32

In [ ]:
if cache_samples:
    # Try to load paired data from pickle file
    if os.path.exists(f'data/{dataset}_{setting}_paired.pkl'):
        with open(f'data/{dataset}_{setting}_paired.pkl', 'rb') as f:
            paired_data = pickle.load(f)
            paired_train = paired_data['train']
            paired_test = paired_data['test']
            print('Paired data loaded from pickle file')
    else:
        # Load data
        X = load_data(dataset, T)
        # Generate paired data and save to pickle file
        paired_train, paired_test = get_cross_data(X, dataset, setting, batch_size, return_loader=False, train_samples=train_samples, test_samples=test_samples)
        if save_samples:
            with open(f'data/{dataset}_{setting}_paired.pkl', 'wb') as f:
                pickle.dump({'train': paired_train, 'test': paired_test}, f)
                print('Paired data saved to pickle file')

else:
    # Load data
    X = load_data(dataset, T)
    paired_train, paired_test = get_cross_data(X, dataset, setting, batch_size, return_loader=False, train_samples=train_samples, test_samples=test_samples)

    if save_samples:
        with open(f'data/{dataset}_{setting}_paired.pkl', 'wb') as f:
            pickle.dump({'train': paired_train, 'test': paired_test}, f)
            print('Paired data saved to pickle file')

In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(paired_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(paired_test, batch_size=batch_size, shuffle=False)

In [ ]:
# Initialize the model
model = Model(num_class=120 if dataset=='ntu120' else 60, num_point=V, num_person=M, graph='graph.ntu_rgb_d.Graph',
              graph_args={'labeling_mode': 'spatial'}, debug=False, dataset=dataset)
model = model.cuda()

# Define optimizer
# Only optimize parameters that require gradients (unfrozen parameters)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

# Number of epochs
num_epochs = 100

# Create Trainer instance
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    train_paired_loader=train_loader,
    val_paired_loader=test_loader,
    num_epochs=num_epochs,
    wandb_project='Motion Retargeting',
    device='cuda', 
    dataset=dataset,
)

# Train
trainer.train()

# Save model
torch.save(model.state_dict(), 'model.pth')